In [2]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import torch
import random
from torch import nn
from torch import optim


In [3]:
def generate_random_walks(G, num_walks=10, walk_length=80):
    walks = []
    nodes = list(G.nodes())
    for _ in range(num_walks):
        random.shuffle(nodes)
        node=nodes[0]
        walk = [node]
        for _ in range(walk_length - 1):
                curr = walk[-1]
                neighbors = list(G.neighbors(curr))
                if neighbors: walk.append(random.choice(neighbors))
                else: walk.append(curr)
        walks.append(walk)
    return walks

In [4]:
def generate_training_pairs(nodes, walks, window_size=1, num_neg_samples=5):
    vocab_size, pairs = len(nodes), []
    for walk in walks:
        for i, center in enumerate(walk):
            start = max(0, i - window_size)
            end = min(len(walk), i + window_size + 1)
            for j in range(start, end):
                if i != j:
                    pos_ctx = walk[j]
                    pairs.append((center, pos_ctx, 1))  # Positive pair
                    for _ in range(num_neg_samples):
                        neg_ctx = random.randint(0, vocab_size - 1)
                        while neg_ctx == center or neg_ctx == pos_ctx:
                            neg_ctx = random.randint(0, vocab_size - 1)
                        pairs.append((center, neg_ctx, 0))  # Negative pair
    return window_size+num_neg_samples,pairs

In [5]:
class GraphEmbedder(nn.Module):
    def __init__(self, num_nodes, embedding_dim):
        super(GraphEmbedder, self).__init__()
        self.embeddings = nn.Embedding(num_nodes, embedding_dim)

    def forward(self, node_ids):
        return self.embeddings(node_ids)

def contrastive_loss(pos_scores, neg_scores):
    loss = 0
    if len(pos_scores) > 0: loss += -torch.log(torch.sigmoid(pos_scores) + 1e-15).sum()
    if len(neg_scores) > 0: loss += torch.log(torch.sigmoid(neg_scores) + 1e-15).sum()
    return loss

In [6]:
G = nx.karate_club_graph()
num_nodes = G.number_of_nodes()
num_nodes
walks = generate_random_walks(G, num_walks=7, walk_length=5)
walks
nodes=list(G.nodes())
batch_size,pairs = generate_training_pairs(nodes, walks)
batch_size
embedding_dim, lr = 3, 1e-2
model = GraphEmbedder(num_nodes, embedding_dim) # .to(device)
optimizer = optim.SGD(model.parameters(), lr=lr)

In [7]:
pairs = np.array(pairs)  # (N, 3): [center, context, label]
centers = torch.tensor(pairs[:, 0], dtype=torch.long) #.to(device)
contexts = torch.tensor(pairs[:, 1], dtype=torch.long) #.to(device)
labels = torch.tensor(pairs[:, 2], dtype=torch.float) #.to(device)

In [8]:
model.train()
num_epochs=1
for epoch in range(num_epochs):
  total_loss = 0.0
  for i in range(0, len(pairs), batch_size):
    batch_centers = centers[i:i+batch_size]
    batch_contexts = contexts[i:i+batch_size]
    batch_labels = labels[i:i+batch_size]

    optimizer.zero_grad()
    center_embeds = model(batch_centers)
    context_embeds = model(batch_contexts)
    # print('c:',center_embeds)
    # print('ct:',context_embeds)

    scores = torch.sum(center_embeds * context_embeds, dim=1)  # (B,)
    # print('c*ct:',scores)

    pos_scores = scores[batch_labels == 1]
    neg_scores = scores[batch_labels == 0]
    # print('pos:',pos_scores)
    # print('neg:',neg_scores)

    loss = contrastive_loss(pos_scores,neg_scores)
    # print('loss:',loss)

    loss.backward()
    optimizer.step()

    total_loss += loss.item()

In [9]:
embedding_dim, lr = 3, 1e-2
model = GraphEmbedder(num_nodes, embedding_dim) # .to(device)
optimizer = optim.SGD(model.parameters(), lr=lr)

walks = generate_random_walks(G, num_walks=1000, walk_length=10)
nodes=list(G.nodes())
batch_size,pairs = generate_training_pairs(nodes, walks)
pairs

model.train()
num_epochs=10
for epoch in range(num_epochs):
  total_loss = 0.0
  for i in range(0, len(pairs), batch_size):
    batch_centers = centers[i:i+batch_size]
    batch_contexts = contexts[i:i+batch_size]
    batch_labels = labels[i:i+batch_size]

    optimizer.zero_grad()
    center_embeds = model(batch_centers)
    context_embeds = model(batch_contexts)
    # print('c:',center_embeds)
    # print('ct:',context_embeds)

    scores = torch.sum(center_embeds * context_embeds, dim=1)  # (B,)
    # print('c*ct:',scores)

    pos_scores = scores[batch_labels == 1]
    neg_scores = scores[batch_labels == 0]
    # print('pos:',pos_scores)
    # print('neg:',neg_scores)

    loss = contrastive_loss(pos_scores,neg_scores)
    # print('loss:',loss)

    loss.backward()
    optimizer.step()

    total_loss += loss.item()
  print(f'Epoch {epoch + 1}/{num_epochs}, Loss: {total_loss / len(pairs)}')

AttributeError: 'int' object has no attribute 'backward'

In [10]:
model.eval()
with torch.no_grad():
  ebeddings = model.embeddings.weight.cpu().data.numpy()

In [11]:
from sklearn.mixture import GaussianMixture
gm_opt,gm_opt_bic,gm_opt_c,gm_opt_n=[],1e100,[],-1
for n in range(1,20):
 clf=GaussianMixture(n_components=n)
 c=clf.fit_predict(ne)
 bic=clf.bic(ne)
 if bic<gm_opt_bic:
  gm_opt=clf
  gm_opt_bic=bic
  gm_opt_c=c
  gm_opt_n=n
plt.scatter(ne[:,0],ne[:,1],c=gm_opt_c)
plt.title(str(gm_opt_n))

NameError: name 'ne' is not defined

In [ ]:
import random
import numpy as np
from collections import defaultdict

def get_alias_edge(G, p, q, src, dst):
    unnormalized_probs = []
    neighbors = sorted(G.neighbors(dst))
    for dst_nbr in neighbors:
        if dst_nbr == src:             unnormalized_probs.append(1/p)
        elif G.has_edge(dst_nbr, src): unnormalized_probs.append(1.0)
        else:                          unnormalized_probs.append(1/q)
    return create_alias_table(unnormalized_probs)

def generate_node2vec_walks(G, num_walks=10, walk_length=80, p=1.0, q=1.0):
    walks = []
    nodes = list(G.nodes())

    alias_nodes = {}
    alias_edges = {}
    # 1. Compute node transition probabilities (normalized 1st order)
    for node in nodes:
        probs = [1.0 for nbr in sorted(G.neighbors(node))]
        alias_nodes[node] = create_alias_table(probs)

    # 2. Compute edge transition probabilities (2nd order, depends on p/q)
    for node in nodes:
        for nbr in G.neighbors(node):
            key = (node, nbr)
            alias_edges[key] = get_alias_edge(G, p, q, node, nbr)

    # 3. Generate walks
    for _ in range(num_walks):
        random.shuffle(nodes)
        for start_node in nodes:
            walk = [start_node]
            walk=[]
            walk.extend(node2vec_walk(G, walk_length - 1, start_node, alias_nodes, alias_edges, p, q))
            walks.append(walk)

    return walks

def node2vec_walk(G, walk_length, start_node, alias_nodes, alias_edges, p=1.0, q=1.0):
    """
    Simulate a single node2vec random walk starting from start_node.
    """
    walk = [start_node]
    prev = -1  # dummy for first step
    curr = start_node

    for _ in range(walk_length - 1):
        curr_neighbors = list(G.neighbors(curr))
        if len(curr_neighbors) == 0:
            break

        if len(walk) == 1:  # First step: only depends on node weights
            next_node = alias_sample(alias_nodes[curr][0], alias_nodes[curr][1])
            next_node = curr_neighbors[next_node]
            walk.append(next_node)
            prev = curr
            curr = next_node
        else:
            prev = walk[-2]
            curr = walk[-1]
            key = (prev, curr)
            # Sample based on 2nd-order transition probabilities
            next_index = alias_sample(alias_edges[key][0], alias_edges[key][1])
            next_node = list(G.neighbors(curr))[next_index]
            walk.append(next_node)

    return walk


def create_alias_table(probs):
    if len(probs) == 0: return [], []

    probs = np.array(probs)
    probs = probs / probs.sum()  # normalize
    n = len(probs)

    J = np.zeros(n, dtype=np.int32)
    q = np.zeros(n)

    # Scale probabilities
    smaller = []
    larger = []
    for i, prob in enumerate(probs):
        q[i] = n * prob
        if q[i] < 1.0: smaller.append(i)
        else: larger.append(i)

    # Construct alias table
    while len(smaller) > 0 and len(larger) > 0:
        small = smaller.pop()
        large = larger.pop()

        J[small] = large
        q[large] = q[large] + q[small] - 1.0

        if q[large] < 1.0:
            smaller.append(large)
        else:
            larger.append(large)

    return J.tolist(), q.tolist()